In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "healthcare_medallion_dbw"

In [0]:
patients_bronze = spark.table(
    f"{CATALOG}.bronze.patients"
)

patients_silver = (
    patients_bronze

    .dropDuplicates(["patient_id"])

    .withColumn(
        "first_name",
        F.initcap(F.trim(F.col("first_name")))
    )

    .withColumn(
        "last_name",
        F.initcap(F.trim(F.col("last_name")))
    )

    .withColumn(
        "gender",
        F.upper(F.trim(F.col("gender")))
    )

    .withColumn(
        "date_of_birth",
        F.to_date("date_of_birth")
    )

    .withColumn(
        "registration_date",
        F.to_date("registration_date")
    )

    .withColumn(
        "email",
        F.lower(F.trim(F.col("email")))
    )

    .withColumn(
        "contact_number_hash",
        F.sha2(
            F.col("contact_number").cast("string"),
            256
        )
    )

    .withColumn(
        "date_of_birth_hash",
        F.sha2(
            F.col("date_of_birth").cast("string"),
            256
        )
    )

    .withColumn(
        "_dq_passed",
        (
            F.col("patient_id").isNotNull()
            &
            F.col("date_of_birth").isNotNull()
            &
            F.col("email").isNotNull()
        )
    )

    .withColumn(
        "_dq_score",
        (
            F.when(F.col("patient_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("date_of_birth").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("email").isNotNull(), 1).otherwise(0)
        ) / F.lit(3.0)
    )

    .withColumn(
        "_dq_failure_reason",
        F.when(
            F.col("_dq_passed") == False,
            F.lit("Mandatory field validation failed")
        )
    )

    .withColumn(
        "_silver_load_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_silver_batch_id",
        F.expr("uuid()")
    )

    .withColumn(
        "_bronze_batch_id",
        F.col("_batch_id")
    )

    .withColumn(
        "_is_current",
        F.lit(True)
    )

    .withColumn(
        "_effective_from",
        F.current_timestamp()
    )

    .withColumn(
        "_effective_to",
        F.lit(None).cast("timestamp")
    )

    .withColumn(
        "_record_version",
        F.lit(1)
    )

    .withColumn(
        "_masked_fields",
        F.lit(
            "contact_number,date_of_birth"
        )
    )

    .withColumn(
        "_enrichment_source",
        F.lit("bronze.patients")
    )
)

In [0]:
(
    patients_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.patients"
    )
)

In [0]:
doctors_bronze = spark.table(
    f"{CATALOG}.bronze.doctors"
)

doctors_silver = (
    doctors_bronze

    .dropDuplicates(["doctor_id"])

    .withColumn(
        "first_name",
        F.initcap(F.trim(F.col("first_name")))
    )

    .withColumn(
        "last_name",
        F.initcap(F.trim(F.col("last_name")))
    )

    .withColumn(
        "specialization",
        F.initcap(F.trim(F.col("specialization")))
    )

    .withColumn(
        "hospital_branch",
        F.initcap(F.trim(F.col("hospital_branch")))
    )

    .withColumn(
        "email",
        F.lower(F.trim(F.col("email")))
    )

    .withColumn(
        "_dq_passed",
        (
            F.col("doctor_id").isNotNull()
            &
            F.col("specialization").isNotNull()
        )
    )

    .withColumn(
        "_dq_score",
        (
            F.when(F.col("doctor_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("specialization").isNotNull(), 1).otherwise(0)
        ) / F.lit(2.0)
    )

    .withColumn(
        "_silver_load_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_silver_batch_id",
        F.expr("uuid()")
    )

    .withColumn(
        "_bronze_batch_id",
        F.col("_batch_id")
    )

    .withColumn(
        "_enrichment_source",
        F.lit("bronze.doctors")
    )
)

In [0]:
(
    doctors_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.doctors"
    )
)

In [0]:
spark.sql(
    f"SHOW TABLES IN {CATALOG}.silver"
).show(truncate=False)

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|silver  |doctors  |false      |
|silver  |patients |false      |
+--------+---------+-----------+



In [0]:
print(
    "patients:",
    spark.table(
        f"{CATALOG}.silver.patients"
    ).count()
)

print(
    "doctors:",
    spark.table(
        f"{CATALOG}.silver.doctors"
    ).count()
)

patients: 50
doctors: 10
